# 06. Population Based Training (PBT) and DARTS Neural Architecture Search

- **Category:** 09_causal_ai_and_evolutionary_search
- **Lineage & Citations:** Jaderberg et al. (2017 *Population Based Training of Neural Networks*); Falkner et al. (2018 *BOHB*); Liu, Simonyan & Yang (2019 *DARTS: Differentiable Architecture Search*).
- **Purpose:** Automate meta-optimization of dynamic hyperparameter schedules and neural network topology in a single training run without training combinatorial candidates from scratch.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [05. Quality Diversity MAP-Elites](05_quality_diversity_map_elites_and_differentiable_qd.ipynb) $\to$ `🔗 Downstream Syntheses:` [07. Causal Representation Identifiability iVAE](07_causal_representation_identifiability_ivae_synthetic_control_and_symbolic_sr.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do not run unregularized DARTS without skip-connection regularization or operation dropout (identity skip connections collapse search by monopolizing gradient flow); do not use PBT when population size $P < 4$ or when training compute cannot support concurrent workers.

## Mathematical Foundations & Governing Equations

1. **PBT Exploit & Explore**: $w_{\text{bot}} \leftarrow w_{\text{top}}, \quad h_{\text{bot}} \leftarrow h_{\text{top}} \odot \boldsymbol{\gamma}, \quad \gamma_m \in \{0.8, 1.2\}$
2. **DARTS Softmax Mixture**: $\bar{o}^{(i, j)}(x) = \sum_{o \in \mathcal{O}} \frac{\exp(\alpha_o^{(i, j)})}{\sum_{o'} \exp(\alpha_{o'}^{(i, j)})} o(x)$
3. **Discrete Decoding**: $o^*(i, j) = \arg\max_{o \in \mathcal{O}} \alpha_o^{(i, j)}$
4. **Architecture Entropy**: $H(\alpha) = -\sum_{e, m} w_{e, m} \log(w_{e, m} + \epsilon)$

In [ ]:
import sys, time
from pathlib import Path
import torch
from safetensors.torch import load_file

r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
if str(r) not in sys.path: sys.path.insert(0, str(r))
w_path = r / "weights/09_causal_ai_and_evolutionary_search/06_population_based_training_pbt_and_darts_neural_architecture_search.safetensors"
weights = load_file(str(w_path))
print(f"Loaded {len(weights)} tensors from {w_path.name}")

In [ ]:
def pbt_step(scores, weights_pop, hyperparams_pop, perturb):
    top, bot = int(torch.argmax(scores)), int(torch.argmin(scores))
    w_new, h_new = weights_pop.clone(), hyperparams_pop.clone()
    w_new[bot] = weights_pop[top]
    h_new[bot] = hyperparams_pop[top] * perturb
    return w_new, h_new

def darts_forward(alpha, candidate_ops):
    w = torch.softmax(alpha, dim=-1)
    return torch.sum(w.unsqueeze(-1).unsqueeze(-1) * candidate_ops, dim=1), w

def darts_decode(alpha, candidate_ops):
    sel = torch.argmax(alpha, dim=-1)
    return sel.float(), candidate_ops[torch.arange(alpha.shape[0]), sel]

def darts_entropy(weights, eps=1e-8):
    return -torch.sum(weights * torch.log(weights + eps))

In [ ]:
# Parity verification against golden reference tensors
w_up, h_up = pbt_step(weights["pbt_scores"], weights["pbt_weights"], weights["pbt_hyperparams"], weights["pbt_perturb"])
assert torch.max(torch.abs(w_up - weights["pbt_weights_updated_ref"])) < 1e-5, "PBT weights mismatch"
assert torch.max(torch.abs(h_up - weights["pbt_hyperparams_updated_ref"])) < 1e-5, "PBT hyperparams mismatch"

mixed, w_soft = darts_forward(weights["darts_alpha"], weights["darts_candidate_ops"])
assert torch.max(torch.abs(mixed - weights["darts_mixed_out_ref"])) < 1e-5, "DARTS mixed out mismatch"
assert torch.max(torch.abs(w_soft - weights["darts_softmax_weights_ref"])) < 1e-5, "DARTS weights mismatch"

ops, disc = darts_decode(weights["darts_alpha"], weights["darts_candidate_ops"])
assert torch.max(torch.abs(ops - weights["darts_selected_ops_ref"])) < 1e-5, "DARTS ops mismatch"
assert torch.max(torch.abs(disc - weights["darts_discrete_out_ref"])) < 1e-5, "DARTS discrete mismatch"

ent = darts_entropy(w_soft)
assert torch.max(torch.abs(ent - weights["darts_entropy_ref"])) < 1e-5, "DARTS entropy mismatch"
print("All PBT and DARTS parity checks passed with max error < 1e-5!")

In [ ]:
# Benchmarking PBT exploit-explore and DARTS mixed operator
t0 = time.perf_counter()
for _ in range(500):
    _ = pbt_step(weights["pbt_scores"], weights["pbt_weights"], weights["pbt_hyperparams"], weights["pbt_perturb"])
    _, w = darts_forward(weights["darts_alpha"], weights["darts_candidate_ops"])
    _ = darts_decode(weights["darts_alpha"], weights["darts_candidate_ops"])
    _ = darts_entropy(w)
lat_us = (time.perf_counter() - t0) / 500 * 1e6
print(f"Fused PBT + DARTS iteration latency: {lat_us:.2f} us")

In [ ]:
# Failure Mode: DARTS Skip-Connection Collapse
# When parameter-free identity connections transmit unobstructed gradients early in training,
# alpha_skip accumulates disproportionate probability unless entropy regularization is applied.
alpha = torch.tensor([0.0, 0.0, 0.0], requires_grad=True) # [identity, conv3x3, conv5x5]
opt = torch.optim.SGD([alpha], lr=0.5)
# Simulate 10 bilevel gradient updates where identity has 3x gradient flow due to zero parameter attenuation
for _ in range(10):
    w = torch.softmax(alpha, dim=-1)
    # Loss gradient strongly prefers unobstructed identity path in early training
    loss = -(w[0] * 3.0 + w[1] * 1.0 + w[2] * 1.0)
    opt.zero_grad(); loss.backward(); opt.step()
w_collapsed = torch.softmax(alpha, dim=-1)
p_skip = w_collapsed[0].item()
print(f"Unregularized Skip-Connection Probability: {p_skip:.4f} (Collapse threshold > 0.85)")
assert p_skip > 0.85, "Demonstrated skip-connection collapse in unregularized DARTS supernet!"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **PBT** concurrently searches model weights and dynamic hyperparameter trajectories in a single training run.
2. **DARTS** replaces combinatorial discrete architecture search with continuous gradient-based bilevel optimization.
3. **Regularization** (entropy bonuses or operation dropout) prevents degenerate skip-connection collapse.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [06_population_based_training_pbt_and_darts_neural_architecture_search.md](../../markdown/09_causal_ai_and_evolutionary_search/06_population_based_training_pbt_and_darts_neural_architecture_search.md)
- **PyTorch 2.x Module:** [06_population_based_training_pbt_and_darts_neural_architecture_search.py](../../pytorch2/09_causal_ai_and_evolutionary_search/06_population_based_training_pbt_and_darts_neural_architecture_search.py)
- **Functional JAX Module:** [06_population_based_training_pbt_and_darts_neural_architecture_search.py](../../jax/09_causal_ai_and_evolutionary_search/06_population_based_training_pbt_and_darts_neural_architecture_search.py)
- **Native Rust Candle Track:** [06_population_based_training_pbt_and_darts_neural_architecture_search.rs](../../candle/09_causal_ai_and_evolutionary_search/06_population_based_training_pbt_and_darts_neural_architecture_search.rs)
- **Hardware Kernel (CPU / WGSL):** [06_population_based_training_pbt_and_darts_neural_architecture_search.rs](../../kernels/09_causal_ai_and_evolutionary_search/06_population_based_training_pbt_and_darts_neural_architecture_search.rs)
- **Golden SafeTensors:** [06_population_based_training_pbt_and_darts_neural_architecture_search.safetensors](../../weights/09_causal_ai_and_evolutionary_search/06_population_based_training_pbt_and_darts_neural_architecture_search.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 05. Quality Diversity Map Elites and Differentiable Qd](05_quality_diversity_map_elites_and_differentiable_qd.ipynb) | [Master Index](../../README.md#09_causal_ai_and_evolutionary_search) | [Next: 07. Causal Representation Identifiability iVAE Synthetic Control and Symbolic Sr](07_causal_representation_identifiability_ivae_synthetic_control_and_symbolic_sr.ipynb) &rarr;